In [20]:
!pip install -q sentence-transformers faiss-cpu

In [21]:
# Imports, knowledge base construction, and FAISS index
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
import warnings
warnings.filterwarnings("ignore")

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

kb = [str(row[row['answer']]) for _, row in train.iterrows()]

model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f"KB size: {len(kb)}, embedding dim: {kb_embeddings.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KB size: 2000, embedding dim: 384


In [22]:
# Zero-shot classifier and shared row-150 variables
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150[L]) for L in 'ABCDE']
correct_answer_text = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [23]:
# Q1: Zero-shot probability of the correct option on row 150
result_q1 = zs(prompt_150, candidate_labels=labels_150)

score_q1 = next(sc for lbl, sc in zip(result_q1['labels'], result_q1['scores']) if lbl == correct_answer_text)

print(f"Q1: {round(score_q1, 3)}")
for lbl, sc in zip(result_q1['labels'], result_q1['scores']):
    marker = " <- correct" if lbl == correct_answer_text else ""
    print(f"  {sc:.4f}  {lbl[:80]}{marker}")

Q1: 0.384
  0.3844  The butterfly effect is the phenomenon that a small change in the initial condit <- correct
  0.3787  The butterfly effect is the phenomenon that a large change in the initial condit
  0.0927  The butterfly effect is the phenomenon that a small change in the initial condit
  0.0786  The butterfly effect is the phenomenon that a small change in the initial condit
  0.0656  The butterfly effect is the phenomenon that a large change in the initial condit


In [24]:
# Q2: FAISS top-10 retrieval and rank of true document (kb[150])
prompt_emb = model.encode([prompt_150], show_progress_bar=False)
D, I = index.search(prompt_emb, 10)
retrieved_indices = I[0].tolist()

true_kb_idx = 150
rank_q2 = retrieved_indices.index(true_kb_idx) + 1 if true_kb_idx in retrieved_indices else "not in top 10"

print(f"Q2: top-10 indices: {retrieved_indices}")
print(f"Q2: rank of kb[150]: {rank_q2}")

Q2: top-10 indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
Q2: rank of kb[150]: 10


In [25]:
# Q3: Cross-encoder rerank of top-10, rank of true document
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]
ce_scores = cross_encoder.predict([[prompt_150, doc] for doc in docs_10])

ranked_indices = [i for i, _ in sorted(zip(retrieved_indices, ce_scores), key=lambda x: -x[1])]
rank_q3 = ranked_indices.index(true_kb_idx) + 1 if true_kb_idx in ranked_indices else "not present"

print(f"Q3: reranked order: {ranked_indices}")
print(f"Q3: rank of kb[150] after reranking: {rank_q3}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q3: reranked order: [150, 847, 1693, 1906, 1269, 1532, 168, 576, 663, 1701]
Q3: rank of kb[150] after reranking: 1


In [26]:
# Q4: BERT token count of the RAG string built from top-5 docs for row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

prompt_emb_42 = model.encode([prompt_42], show_progress_bar=False)
_, I42 = index.search(prompt_emb_42, 5)
retrieved_42 = I42[0].tolist()

rag_string_42 = f"Context: {' '.join(kb[i] for i in retrieved_42)} Question: {prompt_42}"

bert_tok = AutoTokenizer.from_pretrained('bert-base-uncased')
n_tokens = len(bert_tok(rag_string_42, truncation=False, add_special_tokens=True)['input_ids'])

print(f"Q4: top-5 retrieved indices for row 42: {retrieved_42}")
print(f"Q4: BERT token count (with special tokens): {n_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4: top-5 retrieved indices for row 42: [1581, 1760, 42, 241, 439]
Q4: BERT token count (with special tokens): 216


In [27]:
# Q5: Zero-shot on row 150 augmented with the true context (kb[150])
rag_string_q5 = f"Context: {kb[150]} Question: {prompt_150}"
result_q5 = zs(rag_string_q5, candidate_labels=labels_150)

score_q5 = next(sc for lbl, sc in zip(result_q5['labels'], result_q5['scores']) if lbl == correct_answer_text)

print(f"Q5: {round(score_q5, 3)}")
for lbl, sc in zip(result_q5['labels'], result_q5['scores']):
    marker = " <- correct" if lbl == correct_answer_text else ""
    print(f"  {sc:.4f}  {lbl[:80]}{marker}")

Q5: 0.989
  0.9894  The butterfly effect is the phenomenon that a small change in the initial condit <- correct
  0.0045  The butterfly effect is the phenomenon that a large change in the initial condit
  0.0028  The butterfly effect is the phenomenon that a small change in the initial condit
  0.0017  The butterfly effect is the phenomenon that a small change in the initial condit
  0.0016  The butterfly effect is the phenomenon that a large change in the initial condit


In [28]:
# Q6: Zero-shot on row 150 with adversarial context (kb[999])
rag_string_q6 = f"Context: {kb[999]} Question: {prompt_150}"
result_q6 = zs(rag_string_q6, candidate_labels=labels_150)

score_q6 = next(sc for lbl, sc in zip(result_q6['labels'], result_q6['scores']) if lbl == correct_answer_text)

print(f"Q6: adversarial context (kb[999], first 100 chars): {kb[999][:100]}")
print(f"Q6: {round(score_q6, 3)}")
for lbl, sc in zip(result_q6['labels'], result_q6['scores']):
    marker = " <- correct" if lbl == correct_answer_text else ""
    print(f"  {sc:.4f}  {lbl[:80]}{marker}")

Q6: adversarial context (kb[999], first 100 chars): A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts o
Q6: 0.529
  0.5289  The butterfly effect is the phenomenon that a small change in the initial condit <- correct
  0.4253  The butterfly effect is the phenomenon that a large change in the initial condit
  0.0204  The butterfly effect is the phenomenon that a small change in the initial condit
  0.0187  The butterfly effect is the phenomenon that a large change in the initial condit
  0.0067  The butterfly effect is the phenomenon that a small change in the initial condit


In [29]:
# Q7: Hit rate — correct option string found in top-5 retrieved docs, rows 0–99
prompts_100 = [str(train.iloc[i]['prompt']) for i in range(100)]
prompt_embs_100 = model.encode(prompts_100, show_progress_bar=False)
_, I_batch = index.search(prompt_embs_100, 5)

n_hits = sum(
    any(str(train.iloc[i][train.iloc[i]['answer']]) in kb[j] for j in I_batch[i].tolist())
    for i in range(100)
)

print(f"Q7: hits: {n_hits} / 100")
print(f"Q7: hit rate: {round(100 * n_hits / 100, 1)}%")

Q7: hits: 73 / 100
Q7: hit rate: 73.0%


In [30]:
# Q8: Full RAG pipeline MAP@3 — retrieve top-5, cross-encoder rerank, zero-shot classify, rows 0–19
def map3_score(pred_letters, correct_letter):
    return 1.0 / (pred_letters.index(correct_letter) + 1) if correct_letter in pred_letters else 0.0

map3_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_letter_i = row['answer']
    options_i = {L: str(row[L]) for L in 'ABCDE'}
    labels_i = list(options_i.values())

    _, I_i = index.search(model.encode([prompt_i], show_progress_bar=False), 5)
    docs_5 = [kb[j] for j in I_i[0].tolist()]

    ce_scores_i = cross_encoder.predict([[prompt_i, doc] for doc in docs_5])
    best_doc = docs_5[int(np.argmax(ce_scores_i))]

    res = zs(f"Context: {best_doc} Question: {prompt_i}", candidate_labels=labels_i)

    text_to_letter = {v: k for k, v in options_i.items()}
    top3_letters = [text_to_letter[lbl] for lbl in res['labels'][:3] if lbl in text_to_letter]

    score_i = map3_score(top3_letters, correct_letter_i)
    map3_scores.append(score_i)
    print(f"row {i:>2}: top3={top3_letters}, correct={correct_letter_i}, MAP@3={score_i:.3f}")

print(f"\nQ8: average MAP@3 over rows 0-19: {round(np.mean(map3_scores), 3)}")

row  0: top3=['B', 'D', 'A'], correct=B, MAP@3=1.000
row  1: top3=['A', 'E', 'C'], correct=A, MAP@3=1.000
row  2: top3=['C', 'D', 'B'], correct=C, MAP@3=1.000
row  3: top3=['B', 'D', 'A'], correct=B, MAP@3=1.000
row  4: top3=['A', 'B', 'C'], correct=A, MAP@3=1.000


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


row  5: top3=['B', 'C', 'A'], correct=C, MAP@3=0.500
row  6: top3=['E', 'B', 'D'], correct=E, MAP@3=1.000
row  7: top3=['A', 'B', 'C'], correct=A, MAP@3=1.000
row  8: top3=['A', 'C', 'D'], correct=A, MAP@3=1.000
row  9: top3=['A', 'B', 'C'], correct=A, MAP@3=1.000
row 10: top3=['C', 'A', 'D'], correct=C, MAP@3=1.000
row 11: top3=['B', 'C', 'A'], correct=B, MAP@3=1.000
row 12: top3=['D', 'A', 'B'], correct=D, MAP@3=1.000
row 13: top3=['E', 'D', 'B'], correct=E, MAP@3=1.000
row 14: top3=['E', 'A', 'D'], correct=E, MAP@3=1.000
row 15: top3=['E', 'B', 'C'], correct=E, MAP@3=1.000
row 16: top3=['C', 'B', 'E'], correct=C, MAP@3=1.000
row 17: top3=['C', 'B', 'E'], correct=C, MAP@3=1.000
row 18: top3=['B', 'D', 'A'], correct=B, MAP@3=1.000
row 19: top3=['C', 'D', 'A'], correct=C, MAP@3=1.000

Q8: average MAP@3 over rows 0-19: 0.975
